## Get the unconserved variants

In [55]:
import pandas as pd
import vcfpy
import yaml

# config 
config_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/global80K_config.yaml"
with open(config_path) as conf:
    config = yaml.load(conf, Loader=yaml.FullLoader)
    conf.close()

Write it as tab seperated file
```
ultraconserved_regions_path = '/home/kisa/coding/80K_MPRA/info/ultraconserved_regions/ultraconserved_regions_kudernaetal.csv'
ultraconserved_regions = pd.read_csv(ultraconserved_regions_path, sep=";")
ultraconserved_regions.to_csv(ultraconserved_regions_path, sep="\t", index=False)
```

In [9]:
ultraconserved_regions_path = config['files']['creating']['ultraconserved_regions'] # '/home/kisa/coding/80K_MPRA/info/ultraconserved_regions/ultraconserved_regions_kudernaetal.csv'
ultraconserved_regions = pd.read_csv(ultraconserved_regions_path, sep="\t", header=0, names=['chromosome', 'start', 'stop'])
ultraconserved_regions

,chromosome,start,stop
0,chr10,221213,221234
1,chr10,221287,221320
2,chr10,276473,276500
3,chr10,3776567,3776596
4,chr10,3776597,3776627
...,...,...,...
33363,chrX,154030395,154030417
33364,chrX,154030436,154030456
33365,chrX,154031289,154031329
33366,chrX,155065690,155065711


### Idea: 
1. iterate position of variant (vcf)
2. iterate bed file 
3. if overlab between genomic position and bed file => write out as ultraconserved variant

### Using pyranges and vcfpy
1. Open the bed file and iterate over these rows (with apply)
2. Use read vcfpy and reader.fetch('chrom', start, stop) to find the corresponding variants
3. Write the fetched variants for each region into a ultraconserved.vcf
4. Report the number of variants

In [ ]:
bed_df = pd.read_csv('ultraconserved_regions.bed', sep='\t', header=None, names=['chromosome', 'start', 'stop'])

In [11]:
bed_df = pd.read_csv(ultraconserved_regions_path, sep="\t", header=0, names=['chromosome', 'start', 'stop'])

bed_df.itertuples(index=False)

In [4]:
# resolve length not found in contig header
with open('vcf_with_length.vcf', 'w') as f:
    reader = vcfpy.Reader.from_path(config['files']['final_design']['vcf_file'])

    # Get the header from the reader
    header = reader.header

    # Modify the header to ensure contig lines have 'length' attribute
    for header_line in reader.header.get_lines('contig'):
        if 'length' not in header_line['value']:
            header_line['value']['length'] = 0  # Set a default value

    # Create a new Writer object with the modified header
    writer = vcfpy.Writer.from_stream(f, header)

TypeError: 'ContigHeaderLine' object is not subscriptable

In [17]:


# create the tabix of variant vcf
index_command = f"tabix -p vcf {config['files']['final_design']['vcf_file']}"

# Execute the command using subprocess
subprocess.run(index_command, shell=True)

CompletedProcess(args='tabix -p vcf /home/kisa/coding/80K_MPRA/design_data/design_info/variants.vcf.gz', returncode=0)

In [5]:
reader = vcfpy.Reader.from_path(config['files']['final_design']['vcf_file'])
for record in reader:
    print(record.INFO)
    break

{'Region': ['SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1'], 'REF_ID': ['REF_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1'], 'ALT_ID': ['ALT_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778471|1-2179591-T-C'], 'AF': [0.472883]}


/home/kisa/miniforge3/envs/mobil/lib/python3.9/site-packages/vcfpy/header.py:583: FieldInfoNotFound: Field "length" not found in header line contig=<ID=chr1>
  warnings.warn(
/home/kisa/miniforge3/envs/mobil/lib/python3.9/site-packages/vcfpy/header.py:583: FieldInfoNotFound: Field "length" not found in header line contig=<ID=chr10>
  warnings.warn(
/home/kisa/miniforge3/envs/mobil/lib/python3.9/site-packages/vcfpy/header.py:583: FieldInfoNotFound: Field "length" not found in header line contig=<ID=chr11>
  warnings.warn(
/home/kisa/miniforge3/envs/mobil/lib/python3.9/site-packages/vcfpy/header.py:583: FieldInfoNotFound: Field "length" not found in header line contig=<ID=chr12>
  warnings.warn(
/home/kisa/miniforge3/envs/mobil/lib/python3.9/site-packages/vcfpy/header.py:583: FieldInfoNotFound: Field "length" not found in header line contig=<ID=chr13>
  warnings.warn(
/home/kisa/miniforge3/envs/mobil/lib/python3.9/site-packages/vcfpy/header.py:583: FieldInfoNotFound: Field "length" not f

In [14]:
import vcfpy
import pandas as pd
from concurrent.futures import ProcessPoolExecutor

# Step 1: Open the bed file and iterate over the rows
def process_region(chromosome, start, stop):
    variants = []
    
    # Use read vcfpy and reader.fetch('chrom', start, stop) to find the corresponding variants
    reader = vcfpy.Reader.from_path(config['files']['final_design']['vcf_file'])
    for record in reader.fetch(chromosome, start, stop):
        # modify info column:
        record.INFO['ultraconserved_region'] = [f"{chromosome}:{start}-{stop}"]
        # Write the fetched variants for each region into a ultraconserved.vcf
        variants.append(record)
    
    return variants

def main():
    # Read the bed file
    bed_df = pd.read_csv(config['files']['creating']['ultraconserved_regions'], sep="\t", header=0, names=['chromosome', 'start', 'stop'])
    
    # Use multiple processes for parallelization
    with ProcessPoolExecutor() as executor:
        all_variants = executor.map(process_region, bed_df['chromosome'], bed_df['start'], bed_df['stop'])

    # Flatten the list of variants
    all_variants_list = [variant for region_variants in all_variants for variant in region_variants]
    print(all_variants_list)

    # Step 4: Report the number of variants
    print(f"Total number of variants: {len(all_variants_list)}")
    
    # Write the variants to a new VCF file
    with open(config['files']['creating']['ultraconserved_variants'], 'w') as f:
        reader = vcfpy.Reader.from_path(config['files']['final_design']['vcf_file'])
        writer = vcfpy.Writer.from_stream(f, reader.header)
        for record in all_variants_list:
            writer.write_record(record)

if __name__ == "__main__":
    main()

/home/kisa/miniforge3/envs/mobil/lib/python3.9/site-packages/vcfpy/header.py:583: FieldInfoNotFound: Field "length" not found in header line contig=<ID=chr1>
  warnings.warn(
/home/kisa/miniforge3/envs/mobil/lib/python3.9/site-packages/vcfpy/header.py:583: FieldInfoNotFound: Field "length" not found in header line contig=<ID=chr10>
  warnings.warn(
/home/kisa/miniforge3/envs/mobil/lib/python3.9/site-packages/vcfpy/header.py:583: FieldInfoNotFound: Field "length" not found in header line contig=<ID=chr11>
  warnings.warn(
/home/kisa/miniforge3/envs/mobil/lib/python3.9/site-packages/vcfpy/header.py:583: FieldInfoNotFound: Field "length" not found in header line contig=<ID=chr1>
  warnings.warn(
/home/kisa/miniforge3/envs/mobil/lib/python3.9/site-packages/vcfpy/header.py:583: FieldInfoNotFound: Field "length" not found in header line contig=<ID=chr10>
  warnings.warn(
/home/kisa/miniforge3/envs/mobil/lib/python3.9/site-packages/vcfpy/header.py:583: FieldInfoNotFound: Field "length" not fo

[Record('chr10', 60522233, ['cardiac_neuro_cava_random:ANK3|ENSG00000151150.22|EH38E2902697|10-60522233-G-C'], 'G', [Substitution(type_='SNV', value='C')], None, ['PASS'], {'Region': ['ANK3|ENSG00000151150.22|EH38E2902697_rev_tile1-1'], 'REF_ID': ['REF_ANK3|ENSG00000151150.22|EH38E2902697_rev_tile1-1'], 'ALT_ID': ['ALT_ANK3|ENSG00000151150.22|EH38E2902697_rev_tile1-1_ANK3|ENSG00000151150.22|EH38E2902697|10-60522233-G-C'], 'AF': [6.57921e-06], 'ultraconserved_region': ['chr10:60522224-60522248']}, [], []), Record('chr10', 68154368, ['cardiac_neuro_cava_random:MYPN|ENSG00000138347.17|EH38E2905007|10-68154368-G-T'], 'G', [Substitution(type_='SNV', value='T')], None, ['PASS'], {'Region': ['MYPN|ENSG00000138347.17|EH38E2905007_fwd_tile1-1'], 'REF_ID': ['REF_MYPN|ENSG00000138347.17|EH38E2905007_fwd_tile1-1'], 'ALT_ID': ['ALT_MYPN|ENSG00000138347.17|EH38E2905007_fwd_tile1-1_MYPN|ENSG00000138347.17|EH38E2905007|10-68154368-G-T'], 'AF': [6.57056e-06], 'ultraconserved_region': ['chr10:68154347-6

/home/kisa/miniforge3/envs/mobil/lib/python3.9/site-packages/vcfpy/header.py:583: FieldInfoNotFound: Field "length" not found in header line contig=<ID=chr1>
  warnings.warn(
/home/kisa/miniforge3/envs/mobil/lib/python3.9/site-packages/vcfpy/header.py:583: FieldInfoNotFound: Field "length" not found in header line contig=<ID=chr10>
  warnings.warn(
/home/kisa/miniforge3/envs/mobil/lib/python3.9/site-packages/vcfpy/header.py:583: FieldInfoNotFound: Field "length" not found in header line contig=<ID=chr11>
  warnings.warn(
/home/kisa/miniforge3/envs/mobil/lib/python3.9/site-packages/vcfpy/header.py:583: FieldInfoNotFound: Field "length" not found in header line contig=<ID=chr12>
  warnings.warn(
/home/kisa/miniforge3/envs/mobil/lib/python3.9/site-packages/vcfpy/header.py:583: FieldInfoNotFound: Field "length" not found in header line contig=<ID=chr13>
  warnings.warn(
/home/kisa/miniforge3/envs/mobil/lib/python3.9/site-packages/vcfpy/header.py:583: FieldInfoNotFound: Field "length" not f

### Find if ultraconserved variants have effect
1. get the ultraconserved variant ids
2. use the variant region map to get the alt ids
- not even among the analyzable variants
- check for them in the count table; Do they occur with at least 10 barcodes?

In [41]:
# get the ultraconserved variant ids: 
ultraconserved_variant_ids_lists = [] # record for loop returns list of lists
reader = vcfpy.Reader.from_path(config['files']['creating']['ultraconserved_variants'])
for record in reader:
    ultraconserved_variant_ids_lists.append(record.ID)
    
# flatten list of lists
ultraconserved_variant_ids = [variant for variant_list in ultraconserved_variant_ids_lists for variant in variant_list]
print(f'Number of variants: {len(ultraconserved_variant_ids)}')

Number of variants: 69


2. use the variant region map to get the alt ids

In [51]:
variant_map = pd.read_csv(config['files']['final_design']['variant_table'], sep="\t")
variant_map
ultraconserved_variant_map = variant_map.loc[variant_map['Variant'].isin(ultraconserved_variant_ids)]
ultraconserved_alt_ids = ultraconserved_variant_map['ALT_ID'].to_list()
ultraconserved_variant_map

,Variant,Region,REF_ID,ALT_ID
920,cardiac_neuro_cava_random:CASZ1|ENSG0000013094...,cardiac_neuro_cava_random:CASZ1|ENSG0000013094...,cardiac_neuro_cava_random:REF_CASZ1|ENSG000001...,cardiac_neuro_cava_random:ALT_CASZ1|ENSG000001...
1788,cardiac_neuro_cava_random:AHDC1|ENSG0000012670...,cardiac_neuro_cava_random:AHDC1|ENSG0000012670...,cardiac_neuro_cava_random:REF_AHDC1|ENSG000001...,cardiac_neuro_cava_random:ALT_AHDC1|ENSG000001...
1790,cardiac_neuro_cava_random:AHDC1|ENSG0000012670...,cardiac_neuro_cava_random:AHDC1|ENSG0000012670...,cardiac_neuro_cava_random:REF_AHDC1|ENSG000001...,cardiac_neuro_cava_random:ALT_AHDC1|ENSG000001...
1791,cardiac_neuro_cava_random:AHDC1|ENSG0000012670...,cardiac_neuro_cava_random:AHDC1|ENSG0000012670...,cardiac_neuro_cava_random:REF_AHDC1|ENSG000001...,cardiac_neuro_cava_random:ALT_AHDC1|ENSG000001...
1883,cardiac_neuro_cava_random:AHDC1|ENSG0000012670...,cardiac_neuro_cava_random:AHDC1|ENSG0000012670...,cardiac_neuro_cava_random:REF_AHDC1|ENSG000001...,cardiac_neuro_cava_random:ALT_AHDC1|ENSG000001...
...,...,...,...,...
40989,cardiac_neuro_cava_random:FOXP2|ENSG0000012857...,cardiac_neuro_cava_random:FOXP2|ENSG0000012857...,cardiac_neuro_cava_random:REF_FOXP2|ENSG000001...,cardiac_neuro_cava_random:ALT_FOXP2|ENSG000001...
41754,cardiac_neuro_cava_random:KMT2C|ENSG0000005560...,cardiac_neuro_cava_random:KMT2C|ENSG0000005560...,cardiac_neuro_cava_random:REF_KMT2C|ENSG000000...,cardiac_neuro_cava_random:ALT_KMT2C|ENSG000000...
44254,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,cardiac_neuro_cava_random:REF_ZNF462|ENSG00000...,cardiac_neuro_cava_random:ALT_ZNF462|ENSG00000...
44255,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,cardiac_neuro_cava_random:REF_ZNF462|ENSG00000...,cardiac_neuro_cava_random:ALT_ZNF462|ENSG00000...


In [60]:
bc_mpralm_result = pd.read_csv(config['files']['creating']['toptable_bcMPRAlm'], sep="\t")
bc_mpralm_result
print(f'Number of analyzable variants: {bc_mpralm_result.shape[0]}')
# bc_mpralm_result['variant_id'].to_list()
# filter for all the ultra_conserved variants
ultra_conserved_bc_mpralm = bc_mpralm_result.loc[bc_mpralm_result['variant_id'].isin(ultraconserved_variant_ids)]
ultra_conserved_bc_mpralm
print(f'Number of ultra conserved results in bc mpralm: {ultra_conserved_bc_mpralm.shape[0]}')
ultraconserved_significant_results = ultra_conserved_bc_mpralm.loc[ultra_conserved_bc_mpralm['adj.P.Val'] < 0.05]
print(f'Number of ultra conserved results in bc mpralm: {ultraconserved_significant_results.shape[0]}')
print(ultraconserved_significant_results['variant_id'].to_list())
ultraconserved_significant_results

Number of analyzable variants: 35039
Number of ultra conserved results in bc mpralm: 51
Number of ultra conserved results in bc mpralm: 1
['cardiac_neuro_cava_random:FOXP2|ENSG00000128573.28|EH38E2583028|7-114533428-G-T']


,logFC,AveExpr,t,P.Value,adj.P.Val,B,variant_id
295,0.395437,0.587436,3.816764,0.000148,0.017917,0.565156,cardiac_neuro_cava_random:FOXP2|ENSG0000012857...


Check for them in the count table (current data 56)

In [56]:
mprasnakeflow_results = pd.read_csv(config['files']['creating']['mprasnakeflow_resequencing_count_minthreshold'], sep="\t")
mprasnakeflow_results
ultra_conserved_counts = mprasnakeflow_results.loc[mprasnakeflow_results['name'].isin(ultraconserved_alt_ids)] # 168
ultra_conserved_counts['name'].nunique() # 56 # with resequencing: 57

57

In [48]:
mprasnakeflow_results['name']

0         C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...
1         C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...
2         C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...
3         C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...
4         C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...
                                ...                        
203590    cardiac_neuro_cava_random:ZNF462|ENSG000001481...
203591    cardiac_neuro_cava_random:ZNF462|ENSG000001481...
203592    cardiac_neuro_cava_random:ZNF462|ENSG000001481...
203593    cardiac_neuro_cava_random:ZNF462|ENSG000001481...
203594                                                no_BC
Name: name, Length: 203595, dtype: object

### First try at night to get the ultraconserved variants

In [28]:
def is_in_region(row, chrom, var_pos):
    """Check if var_pos in bed row"""
    if chrom == row['chromosome']:
        if row['start'] < var_pos:
            if var_pos <= row['stop']:
                return True
    return False

In [38]:
def find_overlab_in_ultraconserved(bed_file_df, chrom, var_position):
    """Return chrom start end of the ultraconserved sequence the variant is in"""
    # iterate bed file 
    bed_var_pos = bed_file_df.loc[bed_file_df.apply(lambda row: is_in_region(row, chrom, var_position), axis=1)]
    if bed_var_pos.shape[0] > 0:
        if bed_var_pos.shape[0] >= 2:
            print('Multiple ultra conserved sequences for variant found')
        return bed_var_pos
    return 'NA'

In [46]:
def variant_in_region(row):
    """Check the variant file"""
    reader = vcfpy.Reader.from_path(config['files']['final_design']['vcf_file'])
    for record in reader:
        # get variant position
        chromosome = record.CHROM 
        var_pos = record.POS
        if chromosome == row['chromosome']:
            if row['start'] < var_pos:
                if var_pos <= row['stop']:
                    return True
        # TODO: add how to get the information which variant is among the ultra conserved regions
    return False

In [47]:
bed_var_pos = ultraconserved_regions.loc[ultraconserved_regions.apply(variant_in_region, axis=1)]


/home/kisa/miniforge3/envs/mobil/lib/python3.9/site-packages/vcfpy/header.py:583: FieldInfoNotFound: Field "length" not found in header line contig=<ID=chr1>
  warnings.warn(
/home/kisa/miniforge3/envs/mobil/lib/python3.9/site-packages/vcfpy/header.py:583: FieldInfoNotFound: Field "length" not found in header line contig=<ID=chr10>
  warnings.warn(
/home/kisa/miniforge3/envs/mobil/lib/python3.9/site-packages/vcfpy/header.py:583: FieldInfoNotFound: Field "length" not found in header line contig=<ID=chr11>
  warnings.warn(
/home/kisa/miniforge3/envs/mobil/lib/python3.9/site-packages/vcfpy/header.py:583: FieldInfoNotFound: Field "length" not found in header line contig=<ID=chr12>
  warnings.warn(
/home/kisa/miniforge3/envs/mobil/lib/python3.9/site-packages/vcfpy/header.py:583: FieldInfoNotFound: Field "length" not found in header line contig=<ID=chr13>
  warnings.warn(
/home/kisa/miniforge3/envs/mobil/lib/python3.9/site-packages/vcfpy/header.py:583: FieldInfoNotFound: Field "length" not f

KeyboardInterrupt: 

In [45]:
bed_var_pos

,chromosome,start,stop


In [40]:
counting_overlabs = 0
for record in reader:
    # get variant position
    chromosome = record.CHROM 
    position = record.POS
    bed_df = find_overlab_in_ultraconserved(ultraconserved_regions, chromosome, position)
    if bed_df != 'NA':
        print('found overlab')
        counting_overlabs += 1
        
print(f'Number of overlabs with ultraconserved {counting_overlabs}')

found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found overlab
found 

KeyboardInterrupt: 

In [41]:
counting_overlabs

5586

### Pyranges tutorial
